In [1]:
import requests
import json
from datetime import datetime
from zoneinfo import ZoneInfo
import ast
import operator as op

In [2]:
SYSTEM_PROMPT = """You are helpful assistant, You current Timezone is Asia/Bangkok"""

In [3]:
TOOLS = [
	{
		"type": "function",
		"function": {
			"name": "get_current_datetime",
			"description": "Get the current datetime",
			"parameters": {
				"type": "object",
				"properties": {
					"timezone": {
						"type": "string",
						"description": "The timezone to get the current time for"
					}
				},
				"required": [
					"timezone"
				]
			}
		}
	},{
		"type": "function",
		"function": {
			"name": "calculator",
			"description": "Evaluate one or more mathematical expressions with high precision (supports + - * / ^ % and parentheses)",
			"parameters": {
				"type": "object",
				"properties": {
					"expression": {
						"type": "string",
						"description": "Math expression e.g. (12.5 * 3.2)^2 + 1"
					}
				},
				"required": [
					"timezone"
				]
			}
		}
	}
]

In [6]:
def callLLMApi(messages):
	url = "http://localhost:11434/v1/chat/completions"

	payload = json.dumps({
	"model": "gpt-oss",
	"messages": messages,
	"stream": False,
	"temperature": 0.2,
	"tool_choice": "auto",
	"tools": TOOLS
	})
	headers = {
	'Content-Type': 'application/json'
	}

	response = requests.request("POST", url, headers=headers, data=payload)
	return json.loads(response.text)

def get_current_datetime(timezone):
    return datetime.now(ZoneInfo(timezone))

operators = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
             ast.Div: op.truediv, ast.Pow: op.pow}

def calculator(expression):
    def _eval(node):
        if isinstance(node, ast.Num):
            return node.n
        elif isinstance(node, ast.BinOp):
            return operators[type(node.op)](_eval(node.left), _eval(node.right))
        else:
            raise TypeError(node)
    return _eval(ast.parse(expression, mode='eval').body)


In [7]:
complete = False
messages = [
	{ "role": "system", "content": SYSTEM_PROMPT },
    { "role": "user", "content": "ตอนนี้ที่กรุงเทพ และ อเมริกา กี่โมงแล้ว เวลาต่างกันกี่ชั่วโมง" }
]

while(not complete):
    res = callLLMApi(messages)
    message = res['choices'][0]['message']
    messages.append(message)
    
    if('tool_calls' in message):
        for tool in message['tool_calls']:
            content = ''
            if(tool['function']['name'] == 'get_current_datetime'):
                try:
                    func_call = json.loads(tool['function']['arguments'])
                    timezone = func_call['timezone']
                    print("call tool: ", tool['function']['name'], "with timezone:", timezone)
                    current_time = get_current_datetime(timezone)
                    content = str(current_time)
                except Exception as e:
                    content = 'Error: ' + str(e)
                    print(content)
                
            elif(tool['function']['name'] == 'calculator'):
                try:
                    func_call = json.loads(tool['function']['arguments'])
                    expression = func_call['expression']
                    print("call tool: ", tool['function']['name'], "with expression:", expression)
                    result = calculator(expression)
                    content = str(result)
                except Exception as e:
                    content = 'Error: ' + str(e)
                    print(content)
            messages.append({	
				'role': 'tool',
				'content': content,
				'tool_call_id': tool['id']
			})
    else:
        print("Result:")
        print(message['content'])
        complete = True

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001C41166B3D0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))